In [42]:
!pip install optuna

In [43]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [44]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [45]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')


Training set shape: (537, 8)
Test set shape: (231, 8)


Define the objective function

In [46]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [47]:
def objective(trial):
  # Suggest values for the hyperparameters
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the RandomForestClassifier with the suggested hyperparameters
  clf = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth = max_depth,
      random_state=42
  )

  # Perform 3-fold cross validation and calculate the accuracy
  scores = cross_val_score(
      clf,
      X_train,
      y_train,
      cv = 3,
      scoring = 'accuracy'
  ).mean()

  # Return the score
  return scores

Create study

In [48]:
study = optuna.create_study(
    direction = 'maximize',
    study_name = 'RandomForestClassifier',
    sampler = optuna.samplers.TPESampler()
)

[I 2026-10-22 05:21:04,563] A new study created in memory with name: RandomForestClassifier


Optimize the study

In [49]:
study.optimize(objective, n_trials = 50)

[I 2026-10-22 05:21:07,369] Trial 0 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 80, 'max_depth': 5}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-10-22 05:21:08,925] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 113, 'max_depth': 5}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-10-22 05:21:12,766] Trial 2 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 142, 'max_depth': 19}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-10-22 05:21:16,196] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 144, 'max_depth': 8}. Best is trial 0 with value: 0.7709497206703911.
[I 2026-10-22 05:21:23,211] Trial 4 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 166, 'max_depth': 14}. Best is trial 4 with value: 0.7728119180633147.
[I 2026-10-22 05:21:28,663] Trial 5 finished with value: 0.7653631284916201 and parameters: {'n_estimato

Inspect the study

In [50]:
print(f"Best trial: {study.best_trial}")
print(f"Best trial value: {study.best_trial.value}")
print(f"Best trial params: {study.best_trial.params}")

Best trial: FrozenTrial(number=45, state=<TrialState.COMPLETE: 1>, values=[0.7802607076350093], datetime_start=datetime.datetime(2026, 10, 22, 5, 22, 47, 20368), datetime_complete=datetime.datetime(2026, 10, 22, 5, 22, 48, 671470), params={'n_estimators': 129, 'max_depth': 16}, user_attrs={}, system_attrs={'tpe:relative_params:0': '{"max_depth": 16, "n_estimators": 129}'}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=200, log=False, low=50, step=1), 'max_depth': IntDistribution(high=20, log=False, low=3, step=1)}, trial_id=45, value=None)
Best trial value: 0.7802607076350093
Best trial params: {'n_estimators': 129, 'max_depth': 16}


In [51]:
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

Best value: 0.7802607076350093
Best params: {'n_estimators': 129, 'max_depth': 16}


Train the model with suggested hyperparameters

In [52]:
best_model = RandomForestClassifier(
    **study.best_trial.params,
    random_state = 42
)

In [53]:
best_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=16, n_estimators=129, random_state=42)

In [54]:
yhat = best_model.predict(X_test)

In [55]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, yhat)

0.7445887445887446

Visualization

In [72]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [57]:
plot_optimization_history(study).show()

In [58]:
plot_parallel_coordinate(study).show()

In [59]:
plot_param_importances(study).show()

In [60]:
plot_contour(study, params=['n_estimators', 'max_depth']).show()

In [61]:
plot_slice(study, params=['n_estimators', 'max_depth']).show()

Samplers in Optuna - RandomizesSearchCV

In [63]:
def objective(trial):
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  clf = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  scores = cross_val_score(
      clf,
      X_train,
      y_train,
      cv = 3,
      scoring = 'accuracy'
  ).mean()

  return scores


Create study

In [64]:
study = optuna.create_study(
    direction = 'maximize',
    study_name = "RandomForestClassifier - RandomizedSearchCV",
    sampler = optuna.samplers.RandomSampler()
)
study.optimize(objective, n_trials = 50)

[I 2026-10-22 05:23:48,883] A new study created in memory with name: RandomForestClassifier - RandomizedSearchCV
[I 2026-10-22 05:23:50,672] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 97, 'max_depth': 15}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-22 05:23:51,974] Trial 1 finished with value: 0.7579143389199254 and parameters: {'n_estimators': 135, 'max_depth': 3}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-22 05:23:52,609] Trial 2 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 63, 'max_depth': 4}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-22 05:23:54,641] Trial 3 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 149, 'max_depth': 6}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-10-22 05:23:57,270] Trial 4 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 199, 'max_depth': 7}. Best is trial 0 with value: 0.76908752327

Inspect Study

In [65]:
print(f"Best trial: {study.best_trial}")
print(f"Best trial value: {study.best_trial.value}")
print(f"Best trial params: {study.best_trial.params}")
print("-"*45)
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

Best trial: FrozenTrial(number=9, state=<TrialState.COMPLETE: 1>, values=[0.7783985102420857], datetime_start=datetime.datetime(2026, 10, 22, 5, 24, 5, 33419), datetime_complete=datetime.datetime(2026, 10, 22, 5, 24, 5, 843852), params={'n_estimators': 75, 'max_depth': 17}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=200, log=False, low=50, step=1), 'max_depth': IntDistribution(high=20, log=False, low=3, step=1)}, trial_id=9, value=None)
Best trial value: 0.7783985102420857
Best trial params: {'n_estimators': 75, 'max_depth': 17}
---------------------------------------------
Best value: 0.7783985102420857
Best params: {'n_estimators': 75, 'max_depth': 17}


Best Model

In [66]:
best_model = RandomForestClassifier(
    **study.best_trial.params,
    random_state = 42
)

best_model.fit(X_train, y_train)
yhat = best_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, yhat)}")

Accuracy: 0.7619047619047619


> RandomSampler Outperforms TPESampler

In [67]:
plot_optimization_history(study).show()

In [68]:
plot_parallel_coordinate(study).show()

In [69]:
plot_param_importances(study).show()

In [70]:
plot_contour(study, params=['n_estimators', 'max_depth']).show()

In [74]:
plot_slice(study, params=['n_estimators', 'max_depth']).show()

Optimizing Multiple ML Models

In [75]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [84]:
def objective(trial):
  classifier_name = trial.suggest_categorical('classifier', ['RandomForest', 'GradientBoosting', 'SVM'])

  if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

  elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

  elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

  # Perform cross-validation and return the mean accuracy
  score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
  return score

In [80]:
!pip install cmaes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 4.2 MB/s eta 0:00:00


In [85]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(
    direction='maximize',
    study_name = 'Optimizing Multiple ML Models',
    sampler = optuna.samplers.CmaEsSampler()
)
study.optimize(objective, n_trials=100)

[I 2026-10-22 05:39:31,380] A new study created in memory with name: Optimizing Multiple ML Models
[I 2026-10-22 05:39:31,445] Trial 0 finished with value: 0.7337057728119181 and parameters: {'classifier': 'SVM', 'C': 1.9679286775148253, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7337057728119181.
[W 2026-10-22 05:39:31,450] The parameter `classifier` in Trial#1 is sampled independently using `RandomSampler` instead of `CmaEsSampler`, potentially degrading the optimization performance. This fallback happened because dynamic search space and `CategoricalDistribution` are not supported by `CmaEsSampler`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `CmaEsSampler` if this independent sampling is intended behavior.
[W 2026-10-22 05:39:31,452] The parameter `n_estimators` in Trial#1 is sampled independently using `RandomSampler` instead of `CmaEsSampler`, potentially degrading the optimization performance. Th

In [88]:
print(f"Best trial: {study.best_trial}")
print(f"Best trial value: {study.best_trial.value}")
print(f"Best trial params: {study.best_trial.params}")

Best trial: FrozenTrial(number=90, state=<TrialState.COMPLETE: 1>, values=[0.7858472998137803], datetime_start=datetime.datetime(2026, 10, 22, 5, 41, 8, 414660), datetime_complete=datetime.datetime(2026, 10, 22, 5, 41, 8, 462648), params={'classifier': 'SVM', 'C': 0.31719748239582546, 'kernel': 'linear', 'gamma': 'scale'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'classifier': CategoricalDistribution(choices=('RandomForest', 'GradientBoosting', 'SVM')), 'C': FloatDistribution(high=100.0, log=True, low=0.1, step=None), 'kernel': CategoricalDistribution(choices=('linear', 'rbf', 'poly', 'sigmoid')), 'gamma': CategoricalDistribution(choices=('scale', 'auto'))}, trial_id=90, value=None)
Best trial value: 0.7858472998137803
Best trial params: {'classifier': 'SVM', 'C': 0.31719748239582546, 'kernel': 'linear', 'gamma': 'scale'}


In [89]:
print(f"Best value: {study.best_value}")
print(f"Best params: {study.best_params}")

Best value: 0.7858472998137803
Best params: {'classifier': 'SVM', 'C': 0.31719748239582546, 'kernel': 'linear', 'gamma': 'scale'}


In [91]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_cma:generation,state
0,0,0.733706,2026-10-22 05:39:31.385264,2026-10-22 05:39:31.444930,0 days 00:00:00.059666,1.967929,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.761639,2026-10-22 05:39:31.447287,2026-10-22 05:39:34.693844,0 days 00:00:03.246557,NaN,NaN,GradientBoosting,NaN,NaN,0.043740,8.0,8.0,7.0,222.0,0.0,COMPLETE
2,2,0.757914,2026-10-22 05:39:34.695086,2026-10-22 05:39:34.746711,0 days 00:00:00.051625,5.984891,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.735568,2026-10-22 05:39:34.747965,2026-10-22 05:39:34.790354,0 days 00:00:00.042389,3.390438,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.774674,2026-10-22 05:39:34.791527,2026-10-22 05:39:35.576603,0 days 00:00:00.785076,NaN,False,RandomForest,NaN,NaN,NaN,13.0,2.0,2.0,127.0,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.743017,2026-10-22 05:41:10.521058,2026-10-22 05:41:14.096811,0 days 00:00:03.575753,NaN,NaN,GradientBoosting,NaN,NaN,0.102218,19.0,2.0,2.0,102.0,NaN,COMPLETE
96,96,0.756052,2026-10-22 05:41:14.097956,2026-10-22 05:41:18.349863,0 days 00:00:04.251907,NaN,NaN,GradientBoosting,NaN,NaN,0.010464,20.0,5.0,8.0,178.0,NaN,COMPLETE
97,97,0.731844,2026-10-22 05:41:18.353498,2026-10-22 05:41:19.841755,0 days 00:00:01.488257,NaN,NaN,GradientBoosting,NaN,NaN,0.278529,13.0,10.0,4.0,112.0,NaN,COMPLETE
98,98,0.677840,2026-10-22 05:41:19.843121,2026-10-22 05:41:19.966267,0 days 00:00:00.123146,65.053245,NaN,SVM,auto,poly,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [94]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,42
RandomForest,30
GradientBoosting,28


In [95]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.749268
RandomForest,0.766791
SVM,0.739026


In [103]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

# 1. Safely copy the winning parameters from your study
best_params = study.best_trial.params.copy()

# 2. Extract and remove the model name so it doesn't break scikit-learn
winning_classifier = best_params.pop('classifier')

print(f"🏆 The winning model selected by Optuna is: {winning_classifier}")

# 3. Dynamically initialize the correct winning model architecture
if winning_classifier == 'SVM':
    best_model = SVC(**best_params, random_state=42)

elif winning_classifier == 'RandomForest':
    best_model = RandomForestClassifier(**best_params, random_state=42)

elif winning_classifier == 'GradientBoosting':
    best_model = GradientBoostingClassifier(**best_params, random_state=42)

# 4. Train the winning model on your complete training set
best_model.fit(X_train, y_train)

# 5. Generate final predictions and print the evaluation score
yhat = best_model.predict(X_test)
final_accuracy = accuracy_score(y_test, yhat)

print(f"🎯 Final Unbiased Test Accuracy: {final_accuracy:.4f}")


🏆 The winning model selected by Optuna is: SVM
🎯 Final Unbiased Test Accuracy: 0.7446


Iris Dataset

In [105]:
! pip install optuna-integration[xgboost]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 3.1 MB/s eta 0:00:00


In [107]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print('-'*56)
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")


[I 2026-10-22 06:05:07,145] A new study created in memory with name: no-name-6c858490-095a-460e-b417-351a91f540d6


[0]	train-mlogloss:0.88773	eval-mlogloss:0.87815
[1]	train-mlogloss:0.72586	eval-mlogloss:0.70944
[2]	train-mlogloss:0.67656	eval-mlogloss:0.66932
[3]	train-mlogloss:0.63661	eval-mlogloss:0.63109
[4]	train-mlogloss:0.54000	eval-mlogloss:0.52649
[5]	train-mlogloss:0.45357	eval-mlogloss:0.42582
[6]	train-mlogloss:0.43676	eval-mlogloss:0.41082
[7]	train-mlogloss:0.41155	eval-mlogloss:0.38059
[8]	train-mlogloss:0.38135	eval-mlogloss:0.34673
[9]	train-mlogloss:0.35049	eval-mlogloss:0.30833
[10]	train-mlogloss:0.33974	eval-mlogloss:0.29325
[11]	train-mlogloss:0.31312	eval-mlogloss:0.26006
[12]	train-mlogloss:0.28631	eval-mlogloss:0.23053
[13]	train-mlogloss:0.27606	eval-mlogloss:0.21926
[14]	train-mlogloss:0.27487	eval-mlogloss:0.21884
[15]	train-mlogloss:0.27363	eval-mlogloss:0.21799
[16]	train-mlogloss:0.26173	eval-mlogloss:0.20352
[17]	train-mlogloss:0.26074	eval-mlogloss:0.20298
[18]	train-mlogloss:0.26074	eval-mlogloss:0.20298
[19]	train-mlogloss:0.26079	eval-mlogloss:0.20276
[20]	train

[I 2026-10-22 06:05:10,113] Trial 0 finished with value: 1.0 and parameters: {'lambda': 7.03696704899017e-08, 'alpha': 0.8729637882394724, 'eta': 0.2546503079459186, 'gamma': 0.07877455511076698, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.7007462482960004, 'colsample_bytree': 0.4784077962774125}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:0.79024	eval-mlogloss:0.77050
[1]	train-mlogloss:0.65052	eval-mlogloss:0.62250
[2]	train-mlogloss:0.54343	eval-mlogloss:0.49732
[3]	train-mlogloss:0.48607	eval-mlogloss:0.44170
[4]	train-mlogloss:0.42741	eval-mlogloss:0.39163
[5]	train-mlogloss:0.42392	eval-mlogloss:0.38995
[6]	train-mlogloss:0.41714	eval-mlogloss:0.38322
[7]	train-mlogloss:0.39619	eval-mlogloss:0.35096
[8]	train-mlogloss:0.38046	eval-mlogloss:0.32973
[9]	train-mlogloss:0.37835	eval-mlogloss:0.32841
[10]	train-mlogloss:0.37830	eval-mlogloss:0.32840
[11]	train-mlogloss:0.37500	eval-mlogloss:0.31973
[12]	train-mlogloss:0.37139	eval-mlogloss:0.31888
[13]	train-mlogloss:0.34932	eval-mlogloss:0.29749
[14]	train-mlogloss:0.34839	eval-mlogloss:0.29702
[15]	train-mlogloss:0.34649	eval-mlogloss:0.29627
[16]	train-mlogloss:0.34649	eval-mlogloss:0.29631
[17]	train-mlogloss:0.34651	eval-mlogloss:0.29628
[18]	train-mlogloss:0.34657	eval-mlogloss:0.29628
[19]	train-mlogloss:0.34677	eval-mlogloss:0.29646
[20]	train

[I 2026-10-22 06:05:11,125] Trial 1 finished with value: 0.9666666666666667 and parameters: {'lambda': 0.0003828511283095817, 'alpha': 0.027875052038043288, 'eta': 0.2790926033737542, 'gamma': 0.17316612236046075, 'max_depth': 5, 'min_child_weight': 8, 'subsample': 0.455025564544148, 'colsample_bytree': 0.9018244894352763}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.06491	eval-mlogloss:1.06564
[1]	train-mlogloss:1.02502	eval-mlogloss:1.02335
[2]	train-mlogloss:0.98785	eval-mlogloss:0.98333
[3]	train-mlogloss:0.95634	eval-mlogloss:0.95183


[I 2026-10-22 06:05:21,495] Trial 48 pruned. Trial was pruned at iteration 4.


[0]	train-mlogloss:1.06775	eval-mlogloss:1.07009
[1]	train-mlogloss:1.03420	eval-mlogloss:1.03438
[2]	train-mlogloss:1.00243	eval-mlogloss:1.00219
[3]	train-mlogloss:0.97330	eval-mlogloss:0.97450


[I 2026-10-22 06:05:21,550] Trial 49 pruned. Trial was pruned at iteration 4.


--------------------------------------------------------
Best trial: {'lambda': 7.03696704899017e-08, 'alpha': 0.8729637882394724, 'eta': 0.2546503079459186, 'gamma': 0.07877455511076698, 'max_depth': 7, 'min_child_weight': 9, 'subsample': 0.7007462482960004, 'colsample_bytree': 0.4784077962774125}
Best accuracy: 1.0


In [108]:
from optuna.visualization import plot_intermediate_values

# 1. Plot intermediate values during the trials
plot_intermediate_values(study).show()